# MLflow Review + Streamlit Introduction
### Review best model → Understand Streamlit → Build a working prototype

---

This notebook covers two things:

| Part | What we do |
|---|---|
| **Part A** | Review what MLflow logged yesterday and confirm the best model |
| **Part B** | Learn Streamlit from scratch — Hello World through to a forecast prototype |

> **Before starting:** make sure `day1_pipeline.ipynb` has been run and the  
> `mlruns/` folder exists in your project directory.

---

---
# Part A: Review MLflow Results

We can query MLflow directly from the notebook using its Python client.

## Step 1 — Connect to MLflow and list all runs

In [4]:
import os
import mlflow
import pandas as pd

# ── Point MLflow to the same folder used in Day 1 ────────────
DATA_DIR   = r"C:\Users\tthem\timeseries-april" # Change this to your own path
MLFLOW_PATH = os.path.join(DATA_DIR, "mlruns")
mlflow.set_tracking_uri("file:///" + MLFLOW_PATH.replace("\\", "/"))

# ── Read all runs from the experiment ────────────────────────
client = mlflow.tracking.MlflowClient()

# Find the experiment by name
experiment = client.get_experiment_by_name("retail_sales_forecasting")

if experiment is None:
    print("Experiment not found. Make sure Day 1 notebook was run first.")
else:
    print(f"Experiment found: {experiment.name}")
    print(f"Experiment ID:    {experiment.experiment_id}")

Experiment found: retail_sales_forecasting
Experiment ID:    770683833170902931


In [5]:
# ── Pull all runs into a DataFrame ───────────────────────────
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"]   # Most recent runs first
)

# Build the results table
rows = []
for run in runs:
    rows.append({
        'run_name':    run.info.run_name,
        'full_run_id': run.info.run_id,
        'start_time':  run.info.start_time,
        'MAE':         run.data.metrics.get('MAE'),
        'RMSE':        run.data.metrics.get('RMSE'),
        'MAPE':        run.data.metrics.get('MAPE'),
        'Bias':        run.data.metrics.get('Bias'),
        'R2':          run.data.metrics.get('R2'),
    })

df_runs = pd.DataFrame(rows).dropna(subset=['MAE']) 

# Keep only the most recent run for each model name
df_runs = df_runs.sort_values('start_time', ascending=False)
df_runs = df_runs.drop_duplicates(subset='run_name', keep='first')

# Sort by MAE for the final display
df_runs = df_runs.sort_values('MAE').reset_index(drop=True) # Choose other metric if you prefer

print("All logged runs — sorted by MAE (best first):")
print()
print(df_runs[['run_name','MAE','RMSE','MAPE','Bias','R2']].to_string(index=False))

All logged runs — sorted by MAE (best first):

          run_name    MAE   RMSE  MAPE    Bias     R2
             ARIMA  95.65 143.68  21.2   -9.29  0.386
   XGBoost (tuned)  96.46 141.81  22.0   -2.29  0.402
XGBoost_tuned_best  96.46 141.81  22.0   -2.29    NaN
      Holt-Winters  97.99 144.15  22.0   -2.87  0.382
XGBoost (baseline) 108.60 155.72  24.8    5.53  0.279
           Prophet 156.53 201.94  30.9 -144.59 -0.212


## Step 2 — Identify and save the best model's Run ID

In [6]:
# ── Get the best run automatically ───────────────────────────
best_run    = df_runs.iloc[0]
BEST_RUN_ID = best_run['full_run_id']
BEST_MODEL  = best_run['run_name']

print(f"Best model:  {BEST_MODEL}")
print(f"MAE:         {best_run['MAE']}") # You can change this to other metrics if you prefer

Best model:  ARIMA
MAE:         95.65


In [ ]:
# ── Save the Run ID to a text file ───────────────────────────
# This file will be read by the Streamlit app

run_id_path = os.path.join(DATA_DIR, "best_run_id.txt")

with open(run_id_path, "w") as f:       # Use "a" to append if you want to keep history
    f.write(BEST_RUN_ID)

print(f"Run ID saved to: {run_id_path}")
print("The Streamlit app will read this file to load the correct model.")

Run ID saved to: C:\Users\tthem\timeseries-april\best_run_id.txt
The Streamlit app will read this file to load the correct model.


---
# Part B: Introduction to Streamlit

## What is Streamlit?

Streamlit is a Python library that turns a plain `.py` script into an interactive web application

### The key distinction to understand

| | What it is | How you use it |
|---|---|---|
| **`streamlit` (the package)** | A Python library installed with pip | `import streamlit as st` inside a `.py` file |
| **`streamlit run` (the command)** | A terminal command that launches your app | `streamlit run app.py` in Command Prompt |

You never run Streamlit inside a notebook cell.  
You write your app in a `.py` file, then launch it from the terminal.

### Why Streamlit for this project?

- Write Python you already know — no new language to learn
- The forecast plot, date picker, and results table are all built-in components
- One command deploys it to the internet via Streamlit Community Cloud

---
## Step 3 — Install Streamlit

In [8]:
# Install Streamlit — run this cell once
# ! pip install streamlit -q

print("Streamlit installed!")
print()
print("To check the version:")
import streamlit
print("Streamlit version:", streamlit.__version__)

Streamlit installed!

To check the version:
Streamlit version: 1.30.0


---
## Step 4 — How a Streamlit app works

A Streamlit app is a normal Python script (`.py` file) that runs from top to bottom  
every time a user interacts with it.

The key commands you need to know:

| Command | What it does | Example |
|---|---|---|
| `st.title()` | Large heading at the top | `st.title("Sales Forecast")` |
| `st.write()` | Write text, tables, or charts | `st.write("Hello World")` |
| `st.date_input()` | Date picker widget | `st.date_input("Select a date")` |
| `st.selectbox()` | Dropdown menu | `st.selectbox("Model", ["XGBoost","ARIMA"])` |
| `st.slider()` | Numeric slider | `st.slider("Days ahead", 1, 30, 7)` |
| `st.button()` | Clickable button | `st.button("Run Forecast")` |
| `st.line_chart()` | Quick line chart | `st.line_chart(df)` |
| `st.pyplot()` | Display a matplotlib chart | `st.pyplot(fig)` |
| `st.dataframe()` | Display a table | `st.dataframe(df)` |
| `st.sidebar` | Move any widget to the left sidebar | `st.sidebar.title("Settings")` |
| `st.download_button()` | Let users download a file | `st.download_button("Download CSV", ...)` |
| `st.spinner()` | Show a loading message | `with st.spinner("Loading..."):` |
| `st.success()` | Green success message | `st.success("Forecast complete!")` |
| `st.error()` | Red error message | `st.error("Something went wrong.")` |

Official documentation: https://docs.streamlit.io/develop/api-reference  
Date input specifically: https://docs.streamlit.io/develop/api-reference/widgets/st.date_input

---
## Step 5 — Hello World example

The next cell writes a minimal working Streamlit app to a file called `hello_world.py`.  
We use this notebook to *generate* the file — then launch it from the terminal.

In [9]:
# ── Write the Hello World app to a .py file ──────────────────
hello_path = os.path.join(DATA_DIR, "hello_world.py")

hello_code = '''import streamlit as st
import pandas as pd
import numpy as np

# ── Title and introduction ────────────────────────────────────
st.title("Hello, Streamlit!")
st.write("This is your first Streamlit app. Every line of Python you already know works here.")

# ── A simple text display ─────────────────────────────────────
st.header("1. Displaying text")
st.write("Use st.write() to show text, numbers, or tables.")
st.write("Today we are building a forecasting app step by step.")

# ── A date input widget ───────────────────────────────────────
st.header("2. Getting input from the user")
selected_date = st.date_input("Pick a date")
st.write(f"You selected: {selected_date}")

# ── A selectbox ───────────────────────────────────────────────
model_choice = st.selectbox("Choose a model", ["XGBoost", "ARIMA", "Prophet", "Holt-Winters"])
st.write(f"You chose: {model_choice}")

# ── A slider ─────────────────────────────────────────────────
days_ahead = st.slider("How many days to forecast?", min_value=1, max_value=30, value=7)
st.write(f"Forecasting {days_ahead} days ahead.")

# ── A button ─────────────────────────────────────────────────
st.header("3. Buttons and actions")
if st.button("Click me"):
    st.success("Button clicked! In the real app, this will run the forecast.")

# ── A simple chart ────────────────────────────────────────────
st.header("4. Charts")
sample_data = pd.DataFrame({
    "Day":   range(1, 31),
    "Sales": np.random.randint(300, 700, 30)
}).set_index("Day")

st.line_chart(sample_data)
st.write("This is a placeholder chart. On Day 3 it will show real forecast data.")
'''

with open(hello_path, "w") as f:
    f.write(hello_code)

print(f"File written to: {hello_path}")
print()
print("To launch it, open Command Prompt and run:")
print(f"  cd {DATA_DIR}")
print(f"  streamlit run hello_world.py")
print()
print("Then open http://localhost:8501 in your browser.")

File written to: C:\Users\tthem\timeseries-april\hello_world.py

To launch it, open Command Prompt and run:
  cd C:\Users\tthem\timeseries-april
  streamlit run hello_world.py

Then open http://localhost:8501 in your browser.


---
## Step 6 — Prototype the forecast logic here first

> **Key principle:** Always build and test your Python logic in a notebook first.  
> Debugging inside Streamlit is slow — you cannot see intermediate values easily.  
> Once the logic works here, moving it to `app.py` is straightforward.

We will now prototype the core forecast function that the Streamlit app will use.

In [10]:
# ── Load the data and best model ─────────────────────────────
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

# Load the feature-engineered dataset
df = pd.read_csv(os.path.join(DATA_DIR, "data", "timeseries_with_features.csv"))
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').set_index('date')
df = df.dropna()

print("Data loaded!")
print("Shape:", df.shape)
print("Date range:", df.index.min().date(), "to", df.index.max().date())

Data loaded!
Shape: (422, 24)
Date range: 2013-02-01 to 2014-03-31


In [11]:
# ── Define the same feature columns used in Day 1 ────────────
FEATURES = [
    'year', 'month', 'day', 'dayofweek', 'quarter', 'week_of_year',
    'is_weekend', 'is_month_start', 'is_month_end',
    'lag_1', 'lag_7', 'lag_14', 'lag_30',
    'rolling_7d_mean', 'rolling_14d_mean', 'rolling_30d_mean', 'rolling_7d_std',
    'dcoilwtico', 'oil_lag_1', 'oil_rolling_7d_mean',
    'is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
]
TARGET = 'unit_sales'

# ── Load the best model from MLflow ──────────────────────────
model_uri = f"runs:/{BEST_RUN_ID}/model"

try:
    model = mlflow.xgboost.load_model(model_uri)
    print(f"Model loaded from MLflow run: {BEST_RUN_ID[:8]}...")
except Exception as e:
    print(f"MLflow load failed: {e}")
    print("Tip: re-run the Day 1 logging cells to save the model.")

MLflow load failed: Failed to download artifacts from path 'model', please ensure that the path is correct.
Tip: re-run the Day 1 logging cells to save the model.


In [12]:
client = mlflow.tracking.MlflowClient()

# List everything saved inside the best run
artifacts = client.list_artifacts(BEST_RUN_ID)

print(f"Artifacts in run: {BEST_RUN_ID[:8]}...")
print()

if len(artifacts) == 0:
    print("No artifacts found — the model was NOT saved in this run.")
    print("This is the cause of the error.")
else:
    for a in artifacts:
        print(f"  {a.path}  ({a.file_size} bytes)")

Artifacts in run: ef2ed25d...

No artifacts found — the model was NOT saved in this run.
This is the cause of the error.


In [11]:
# ── Log the model into the existing best run ──────────────────
# This adds the model file to the run that already has the metrics.

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.xgboost.log_model(best_xgb, artifact_path="model")
    print(f"Model logged into run: {BEST_RUN_ID[:8]}...")
    print("Try loading it again now.")

NameError: name 'best_xgb' is not defined

In [8]:
# ── Core forecast function ────────────────────────────────────
# This is the function the Streamlit app will call.
# We test it here first so we know it works before moving it to app.py.

def make_forecast(df, model, features, cutoff_date, n_days=1):
    """
    Generate a forecast for n_days starting from cutoff_date.

    Parameters:
        df          : the full feature-engineered DataFrame
        model       : the trained model (XGBoost or similar)
        features    : list of feature column names
        cutoff_date : forecast starts from this date (string or datetime)
        n_days      : how many days to forecast (default: 1)

    Returns:
        DataFrame with columns ['date', 'forecast']
    """
    cutoff = pd.to_datetime(cutoff_date)

    # Get the data up to the cutoff date
    history = df.loc[df.index <= cutoff].copy()

    if len(history) == 0:
        raise ValueError(f"No data found on or before {cutoff_date}.")

    forecasts = []

    for i in range(n_days):
        # The next date to forecast
        next_date = cutoff + pd.Timedelta(days=i+1)

        # Check if this date already exists in the data (use its features)
        if next_date in df.index:
            row = df.loc[[next_date], features]
        else:
            # Date not in data — use the last available row's features
            # In a production app we would engineer these properly
            row = history.iloc[[-1]][features].copy()
            row.index = [next_date]

        # Make the prediction
        pred = model.predict(row)[0]
        pred = max(0, pred)   # Sales cannot be negative

        forecasts.append({'date': next_date, 'forecast': round(pred, 2)})

    return pd.DataFrame(forecasts).set_index('date')

print("make_forecast() defined successfully.")

make_forecast() defined successfully.


In [9]:
# ── Test the forecast function ────────────────────────────────

# Single day forecast
cutoff = '2014-01-15'
forecast_1day = make_forecast(df, model, FEATURES, cutoff, n_days=1)
print("Single day forecast:")
print(forecast_1day)
print()

# 7-day forecast
forecast_7day = make_forecast(df, model, FEATURES, cutoff, n_days=7)
print("7-day forecast:")
print(forecast_7day)

NameError: name 'model' is not defined

In [ ]:
# ── Plot the prototype forecast ───────────────────────────────
# This is the same chart the Streamlit app will display.

cutoff     = pd.to_datetime('2014-01-15')
n_days     = 14
history_days = 60   # Show 60 days of history before the cutoff

# History window
history_plot = df.loc[
    (df.index >= cutoff - pd.Timedelta(days=history_days)) &
    (df.index <= cutoff)
][TARGET]

# Forecast
forecast_plot = make_forecast(df, model, FEATURES, cutoff, n_days=n_days)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(history_plot.index, history_plot.values,
        label='Historical sales', color='steelblue', linewidth=1.5)
ax.plot(forecast_plot.index, forecast_plot['forecast'].values,
        label=f'{n_days}-day forecast', color='orange',
        linestyle='--', linewidth=2, marker='o', markersize=4)
ax.axvline(cutoff, color='red', linestyle=':', linewidth=1.5, label='Cutoff date')
ax.set_title(f'Sales Forecast from {cutoff.date()}')
ax.set_ylabel('Unit Sales')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Prototype chart working correctly.")
print("This same chart will appear inside the Streamlit app.")

---
## Step 7 — Write the Streamlit prototype app

Now that the logic works in the notebook, we write it into `app_prototype.py`.  
This is a simple, readable version — Day 3 will polish and expand it.

In [ ]:
# ── Write the prototype app to a .py file ────────────────────
app_path = os.path.join(DATA_DIR, "app_prototype.py")

app_code = f'''# ── Retail Sales Forecasting App — Prototype ─────────────────
# Run from terminal: streamlit run app_prototype.py
# This is the rough prototype built on Day 2.
# Day 3 will add more features, polish, and deploy this to the cloud.

import os
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
from datetime import date

# ── Configuration ─────────────────────────────────────────────
DATA_DIR    = r"{DATA_DIR}"
MLFLOW_PATH = os.path.join(DATA_DIR, "mlruns")
mlflow.set_tracking_uri("file:///" + MLFLOW_PATH.replace("\\\\", "/"))

FEATURES = [
    "year", "month", "day", "dayofweek", "quarter", "week_of_year",
    "is_weekend", "is_month_start", "is_month_end",
    "lag_1", "lag_7", "lag_14", "lag_30",
    "rolling_7d_mean", "rolling_14d_mean", "rolling_30d_mean", "rolling_7d_std",
    "dcoilwtico", "oil_lag_1", "oil_rolling_7d_mean",
    "is_national_holiday", "is_regional_holiday", "is_local_holiday",
]
TARGET = "unit_sales"

# ── Load data (cached so it only runs once) ───────────────────
@st.cache_data
def load_data():
    df = pd.read_csv(os.path.join(DATA_DIR, "data", "timeseries_with_features.csv"))
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").set_index("date").dropna()
    return df

# ── Load model (cached so it only runs once) ──────────────────
@st.cache_resource
def load_model():
    run_id_file = os.path.join(DATA_DIR, "best_run_id.txt")
    with open(run_id_file) as f:
        run_id = f.read().strip()
    model = mlflow.xgboost.load_model(f"runs:/{{run_id}}/model")
    return model

# ── Forecast function (same as notebook prototype) ────────────
def make_forecast(df, model, features, cutoff_date, n_days=1):
    cutoff    = pd.to_datetime(cutoff_date)
    history   = df.loc[df.index <= cutoff].copy()
    forecasts = []
    for i in range(n_days):
        next_date = cutoff + pd.Timedelta(days=i+1)
        if next_date in df.index:
            row = df.loc[[next_date], features]
        else:
            row = history.iloc[[-1]][features].copy()
            row.index = [next_date]
        pred = max(0, model.predict(row)[0])
        forecasts.append({{"date": next_date, "forecast": round(pred, 2)}})
    return pd.DataFrame(forecasts).set_index("date")

# ── App layout ────────────────────────────────────────────────
st.title("Retail Sales Forecasting")
st.write("Corporacion Favorita — Guayas region demand forecast")

# Sidebar controls
st.sidebar.header("Forecast settings")

cutoff_date = st.sidebar.date_input(
    "Cutoff date",
    value=date(2014, 1, 15),
    min_value=date(2013, 6, 1),
    max_value=date(2014, 3, 30),
    help="The forecast starts from the day after this date."
)

n_days = st.sidebar.slider(
    "Days to forecast",
    min_value=1, max_value=30, value=7
)

history_days = st.sidebar.slider(
    "History days to show",
    min_value=14, max_value=120, value=60
)

run_button = st.sidebar.button("Run Forecast")

# ── Main panel ────────────────────────────────────────────────
if run_button:
    with st.spinner("Loading data and model..."):
        df    = load_data()
        model = load_model()

    with st.spinner("Generating forecast..."):
        cutoff     = pd.to_datetime(cutoff_date)
        history_plot = df.loc[
            (df.index >= cutoff - pd.Timedelta(days=history_days)) &
            (df.index <= cutoff)
        ][TARGET]

        forecast_df = make_forecast(df, model, FEATURES, cutoff, n_days)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(history_plot.index, history_plot.values,
            label="Historical sales", color="steelblue", linewidth=1.5)
    ax.plot(forecast_df.index, forecast_df["forecast"].values,
            label=f"{{n_days}}-day forecast", color="orange",
            linestyle="--", linewidth=2, marker="o", markersize=4)
    ax.axvline(cutoff, color="red", linestyle=":", linewidth=1.5, label="Cutoff date")
    ax.set_title(f"Sales Forecast from {{cutoff.date()}}")
    ax.set_ylabel("Unit Sales")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    st.pyplot(fig)

    # Forecast table
    st.subheader("Forecast values")
    st.dataframe(forecast_df.reset_index().rename(
        columns={{"date":"Date","forecast":"Predicted Sales"}}
    ))

    # Download button
    csv = forecast_df.reset_index().to_csv(index=False)
    st.download_button(
        label="Download forecast as CSV",
        data=csv,
        file_name=f"forecast_{{cutoff_date}}.csv",
        mime="text/csv"
    )

    st.success("Forecast complete!")

else:
    st.info("Adjust the settings in the sidebar and click Run Forecast.")
    st.write("**Data range available:** January 2013 – March 2014")
    st.write("**Models available:** whichever performed best on Day 1")
'''

with open(app_path, "w") as f:
    f.write(app_code)

print(f"Prototype app written to: {{app_path}}")
print()
print("To launch it, open Command Prompt and run:")
print(f"  cd {{DATA_DIR}}")
print("  streamlit run app_prototype.py")
print()
print("Then open http://localhost:8501 in your browser.")

---
## Summary — What we built on Day 2

| Step | What we did |
|---|---|
| MLflow review | Queried all logged runs and identified the best model |
| Saved Run ID | Stored the best model's Run ID for the Streamlit app to use |
| Streamlit concepts | Learned the key components — date input, slider, button, chart |
| Hello World | Wrote and launched a minimal working app |
| Prototype logic | Tested the `make_forecast()` function in the notebook first |
| Prototype app | Wrote `app_prototype.py` — a working but simple forecasting app |

---
### What comes next — Day 3

- Add error handling and user-friendly messages to the app
- Add store and product family filters
- Add N-day autoregressive forecasting mode
- Polish the chart and layout
- Create `requirements.txt` and `README.md`
- Push everything to GitHub
- Deploy to Streamlit Community Cloud — the app goes live

---
> **GitHub commit for today:**  
> Add `day2_streamlit_intro.ipynb`, `app_prototype.py`, and `hello_world.py`  
> Commit message: `"Day 2: MLflow review, Streamlit intro, forecast prototype"`